<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/notebooks/jiwoo/05_jw_cls_single_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **0. Overview**

### **single-only (v1)**

- single 12,000장만 사용
- DB item_seq 기준으로 조인하여 레이블(drug_shape, color_class1, form_code_name, line_front) 직접 구성
- 레이블 전처리:
  - drug_shape 희귀 클래스(삼각형/사각형/오각형/육각형) → 기타 통합 → 최종 5개 클래스
  - color_class1 복합값("노랑, 투명") → 앞쪽만 사용
  - form_code_name 희귀 클래스(서방정/발포정/산제) → 기타 통합 → 최종 13개 클래스
  - line_front 이진화(+/-/기타 → "있음", 나머지 → "없음") → 최종 2개 클래스
- item_seq 단위 StratifiedGroupKFold (n_splits=5, drug_shape 기준 stratify) → train 9,605 / val 2,395
- 실시간 전처리 Dataset (AWB + Bilateral + CLAHE 매번 수행) → epoch당 매우 느림
- **ConvNeXt-tiny** 학습 시도
- drug_shape val 0.92, color val 0.74~0.77 과적합

# **1. Import**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile
import pandas as pd
import numpy as np
!pip install pymysql
import pymysql
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
import pickle
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import cv2
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
import torch
import os
from PIL import Image
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 655.1 kB/s eta 0:00:00


# **2. Manifest Load**

In [12]:
# Sol1: single 12,000장
# single+combination 전체 manifest 로드 후 single 필터링
df_all = pd.read_csv('/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/jiwoo/pilliot_15k_v1_full_manifest_raw.csv')
df_single = df_all[df_all['dataset_type'] == 'single'].copy()

print(f"전체: {len(df_all)}행")
print(f"single: {len(df_single)}행")

전체: 23494행
single: 12000행


# **3. Pre-processing + StratifiedGroupKFold Split**

In [13]:
# 전처리
rare_shapes = ['삼각형', '오각형', '육각형', '사각형']
df_single['drug_shape'] = df_single['drug_shape'].replace(rare_shapes, '기타')

df_single['color_class1'] = df_single['color_class1'].str.split(',').str[0].str.strip()

rare_forms = ['산제', '발포정', '서방정']
df_single['form_code_name'] = df_single['form_code_name'].replace(rare_forms, '기타')

df_single['line_front'] = df_single['line_front'].replace('', '없음').fillna('없음')
df_single['line_front'] = df_single['line_front'].apply(
    lambda x: '있음' if str(x).strip() in ['+', '-', '기타'] else '없음'
)

In [14]:
# StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=5)
X = df_single.index
y = df_single['drug_shape']
groups = df_single['item_seq']

train_idx, val_idx = next(sgkf.split(X, y, groups))
df_train = df_single.iloc[train_idx].copy()
df_val = df_single.iloc[val_idx].copy()

print(f"train: {len(df_train)}행 / val: {len(df_val)}행")
print(f"item_seq overlap: {len(set(df_train['item_seq']) & set(df_val['item_seq']))}")

train: 9605행 / val: 2395행
item_seq overlap: 0


In [7]:
# 레이블 인코딩 (train 기준으로 fit, val은 transform만)
label_cols = ['drug_shape', 'color_class1', 'form_code_name', 'line_front']
encoders = {}

for col in label_cols:
    le = LabelEncoder()
    df_train[f'{col}_label'] = le.fit_transform(df_train[col])
    df_val[f'{col}_label'] = le.transform(df_val[col])
    encoders[col] = le
    print(f"[{col}] 클래스 수: {len(le.classes_)}")
    print(dict(zip(le.classes_, range(len(le.classes_)))))
    print()

# 저장
# single 12,000장 / drug_shape 기준 StratifiedGroupKFold / 레이블 전처리 완료
# drug_shape: 5클래스 / color_class1: 15클래스 / form_code_name: 13클래스 / line_front: 2클래스
save_dir = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/jiwoo/'

# df_train.to_csv(save_dir + 'pilliot_split_train.csv', index=False)
# df_val.to_csv(save_dir + 'pilliot_split_val.csv', index=False)

with open(save_dir + 'label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

print("저장 완료")

[drug_shape] 클래스 수: 5
{'기타': 0, '원형': 1, '장방형': 2, '타원형': 3, '팔각형': 4}

[color_class1] 클래스 수: 15
{'갈색': 0, '검정': 1, '노랑': 2, '보라': 3, '분홍': 4, '빨강': 5, '연두': 6, '자주': 7, '주황': 8, '청록': 9, '초록': 10, '투명': 11, '파랑': 12, '하양': 13, '회색': 14}

[form_code_name] 클래스 수: 13
{'경질캡슐제, 과립제': 0, '경질캡슐제, 산제': 1, '구강붕해정': 2, '기타': 3, '나정': 4, '당의정': 5, '서방성캡슐제, 펠렛': 6, '서방성필름코팅정': 7, '연질캡슐제, 액상': 8, '연질캡슐제, 현탁상': 9, '장용성필름코팅정': 10, '추어블정(저작정)': 11, '필름코팅정': 12}

[line_front] 클래스 수: 2
{'없음': 0, '있음': 1}

저장 완료


# **4. Data Loader 생성**


### **전처리 + 로더 (최초 시도)**

- PillDataset.__getitem__에서 매번 실시간으로 AWB → Bilateral Filter → bbox crop → CLAHE 수행
- epoch당 약 40~50분 소요 → 학습 현실적으로 불가능하다고 판단

In [8]:
# 학습 데이터 로더
class PillDataset(Dataset):
    def __init__(self, manifest_path, extract_path, transform=None):
        self.df = pd.read_csv(manifest_path)
        self.extract_path = extract_path
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

         # 압축 해제된 경로에서 직접 읽기
        img_path = os.path.join(self.extract_path, row['zip_path'])

        # ── 1차 전처리 ──────────────────────────────────────────────────

        # [1] 이미지 포맷 정규화
        # PNG(RGBA)인 경우 알파 채널 자동 제거 → RGB 3채널로 통일
        img = Image.open(img_path).convert('RGB')

        # [2] AWB (Auto White Balance, 오토 화이트 밸런스)
        # 노란 전구·파란 형광등 등 촬영 환경에 따른 색온도 편향을 중립적으로 보정
        # PIL RGB → OpenCV BGR로 변환 후 이후 모든 연산은 BGR 기반으로 수행
        img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        img_cv = self._auto_white_balance(img_cv)

        # [3] 노이즈 제거 - 바이래터럴 필터 (Bilateral Filter)
        # 일반 가우시안 블러와 달리 엣지(각인·윤곽선)는 보존하면서 평탄한 배경 노이즈만 제거
        # d=5: 필터 적용 반경, sigmaColor/sigmaSpace: 색상·공간 유사도 허용 범위
        img_cv = cv2.bilateralFilter(img_cv, d=5, sigmaColor=30, sigmaSpace=30)

        # ── bbox crop ───────────────────────────────────────────────────

        # manifest CSV의 bbox 좌표(x, y, w, h)로 알약 영역 crop
        # 10% padding: 알약 테두리가 잘리지 않도록 여백 추가
        # max(0, ...) / min(img_size, ...): 이미지 경계 벗어나지 않도록 클리핑
        x, y, w, h = int(row['bbox_x']), int(row['bbox_y']), int(row['bbox_w']), int(row['bbox_h'])
        pad_x, pad_y = int(w * 0.1), int(h * 0.1)
        ih, iw = img_cv.shape[:2]
        x1 = max(0, x - pad_x)
        y1 = max(0, y - pad_y)
        x2 = min(iw, x + w + pad_x)
        y2 = min(ih, y + h + pad_y)
        cropped = img_cv[y1:y2, x1:x2]

        # ── 2차 전처리 ──────────────────────────────────────────────────

        # [4] CLAHE (Contrast Limited Adaptive Histogram Equalization)
        # 전체 이미지가 아닌 타일 단위로 대비를 조정하여 음각 각인의 그림자를 뚜렷하게 강조
        # RGB 색상 정보 유지를 위해 grayscale 변환 없이 BGR 각 채널에 독립적으로 적용
        # (grayscale 변환은 OCR 전처리용이며 속성 분류기에서는 색상 head 유지를 위해 미적용)
        cropped = self._apply_clahe(cropped)

        # ── 모델 입력 포맷 변환 ─────────────────────────────────────────

        # OpenCV BGR → RGB 변환 후 PIL Image로 변환
        # PyTorch transforms가 PIL Image를 입력으로 받기 때문에 변환 필요
        # 이후 transforms.ToTensor()가 PIL → Tensor로 최종 변환
        cropped = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(cropped)

        if self.transform:
            img_pil = self.transform(img_pil)

        # ── 레이블 반환 ─────────────────────────────────────────────────
        labels = {
            'drug_shape': torch.tensor(row['drug_shape_label'], dtype=torch.long),
            'color_class1': torch.tensor(row['color_class1_label'], dtype=torch.long),
            'form_code_name': torch.tensor(row['form_code_name_label'], dtype=torch.long),
            'line_front': torch.tensor(row['line_front_label'], dtype=torch.long),
        }

        return img_pil, labels

    def _auto_white_balance(self, img):
        # 각 채널(B, G, R)의 평균값을 128(중립)으로 맞추는 간단한 Gray World AWB
        result = img.copy().astype(np.float32)
        for i in range(3):
            channel = result[:, :, i]
            result[:, :, i] = channel * (128.0 / (np.mean(channel) + 1e-6))
        return np.clip(result, 0, 255).astype(np.uint8)

    def _apply_clahe(self, img):
        # clipLimit: 대비 증폭 상한값 (너무 높으면 노이즈 증폭 위험)
        # tileGridSize: 타일 크기 (8x8이 일반적)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        result = img.copy()
        for i in range(3):
            result[:, :, i] = clahe.apply(result[:, :, i])
        return result

### **전처리 후 → 로더**
- 전처리(AWB + Bilateral Filter + bbox crop + CLAHE)를 미리 수행하여 crop 이미지로 저장
- PillDataset.__getitem__에서는 저장된 이미지 읽기만 수행
- epoch당 약 30~35분으로 단축 (Drive I/O 병목은 여전히 존재, epoch 2부터 캐싱 효과로 빠라짐)
- 이 방식으로 이후 모든 실험 진행

In [10]:
# 전처리 및 저장 함수
def preprocess_and_save(df, zip_path, save_dir):
    failed = []
    with zipfile.ZipFile(zip_path, 'r') as z:
        for idx, row in tqdm(df.iterrows(), total=len(df)):
            save_path = os.path.join(save_dir, row['image_file'])
            if os.path.exists(save_path):
                continue
            try:
                with z.open(row['zip_path']) as f:
                    img = Image.open(f).convert('RGB')

                img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

                # AWB
                result = img_cv.copy().astype(np.float32)
                for i in range(3):
                    channel = result[:, :, i]
                    result[:, :, i] = channel * (128.0 / (np.mean(channel) + 1e-6))
                img_cv = np.clip(result, 0, 255).astype(np.uint8)

                # Bilateral Filter
                img_cv = cv2.bilateralFilter(img_cv, d=5, sigmaColor=30, sigmaSpace=30)

                # bbox crop
                x, y, w, h = int(row['bbox_x']), int(row['bbox_y']), int(row['bbox_w']), int(row['bbox_h'])
                pad_x, pad_y = int(w * 0.1), int(h * 0.1)
                ih, iw = img_cv.shape[:2]
                x1, y1 = max(0, x - pad_x), max(0, y - pad_y)
                x2, y2 = min(iw, x + w + pad_x), min(ih, y + h + pad_y)
                cropped = img_cv[y1:y2, x1:x2]

                # CLAHE
                clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
                for i in range(3):
                    cropped[:, :, i] = clahe.apply(cropped[:, :, i])

                # 저장
                cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
                Image.fromarray(cropped_rgb).save(save_path)

            except Exception as e:
                failed.append((row['image_file'], str(e)))

    print(f"완료. 실패: {len(failed)}개")
    return failed

In [11]:
class PillDataset(Dataset):
    def __init__(self, manifest_path, crop_dir, transform=None):
        self.df = pd.read_csv(manifest_path)
        self.crop_dir = crop_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 전처리된 crop 이미지 바로 읽기
        img_path = os.path.join(self.crop_dir, row['image_file'])
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        labels = {
            'drug_shape': torch.tensor(row['drug_shape_label'], dtype=torch.long),
            'color_class1': torch.tensor(row['color_class1_label'], dtype=torch.long),
        }

        return img, labels

In [16]:
# transform 정의
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0),
    # transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


crop_dir = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/jiwoo/cropped_images/'

train_dataset = PillDataset(save_dir + 'pilliot_split_train.csv', crop_dir, transform=train_transform)
val_dataset = PillDataset(save_dir + 'pilliot_split_val.csv', crop_dir, transform=val_transform)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"train: {len(train_dataset)}장 / val: {len(val_dataset)}장")

train: 9605장 / val: 2395장


# **5. ConvNeXt-Tiny**

#### **모델 아키텍처**

- backbone: ConvNeXt-Tiny (ImageNet1K pretrained) feature extractor로 활용, 출력 feature dim 768
- head: 기존 head 제거 후 drug_shape / color_class1 두 속성 동시 분류하는 멀티 헤드 구성

In [17]:
# ConvNeXt Tiny 백본 로드 (pretrained)
backbone = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
backbone.classifier = nn.Identity()

# 전체 head 후보
all_num_classes = {
    'drug_shape': 5,
    'color_class1': 15,
    'form_code_name': 14,  # 새 split 기준
    'line_front': 2,
}

# 멀티 헤드 분류기 아키텍처 정의
class PillClassifier(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        feature_dim = 768

        # num_classes에 있는 head만 생성
        self.heads = nn.ModuleDict({
            name: nn.Linear(feature_dim, n)
            for name, n in num_classes.items()
        })

    def forward(self, x):
        feat = self.backbone(x)
        feat = feat.flatten(1)  # [B, 768, 1, 1] → [B, 768]
        return {name: head(feat) for name, head in self.heads.items()}

model_2head = PillClassifier(backbone, {k: all_num_classes[k] for k in ['drug_shape', 'color_class1']})

In [18]:
# 학습 루프
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model_2head.to(device)  # V1: drug_shape + color_class1

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = {k: 0 for k in model.heads.keys()}
    total = 0

    for i, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(device)
        labels = {k: v.to(device) for k, v in labels.items() if k in model.heads}

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = sum(criterion(outputs[k], labels[k]) for k in model.heads.keys())
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total += imgs.size(0)
        for k in model.heads.keys():
            correct[k] += (outputs[k].argmax(1) == labels[k]).sum().item()

        if i % 10 == 0:
            acc_str = ' | '.join([f"{k}: {correct[k]/total:.4f}" for k in model.heads.keys()])
            print(f"  [train] batch {i}/{len(loader)} | loss: {total_loss/(i+1):.4f} | {acc_str}", flush=True)

    return total_loss / len(loader), {k: correct[k] / total for k in model.heads.keys()}


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = {k: 0 for k in model.heads.keys()}
    total = 0

    with torch.no_grad():
        for i, (imgs, labels) in enumerate(loader):
            imgs = imgs.to(device)
            labels = {k: v.to(device) for k, v in labels.items() if k in model.heads}
            outputs = model(imgs)
            loss = sum(criterion(outputs[k], labels[k]) for k in model.heads.keys())
            total_loss += loss.item()
            total += imgs.size(0)
            for k in model.heads.keys():
                correct[k] += (outputs[k].argmax(1) == labels[k]).sum().item()

            if i % 10 == 0:
                acc_str = ' | '.join([f"{k}: {correct[k]/total:.4f}" for k in model.heads.keys()])
                print(f"  [val]   batch {i}/{len(loader)} | loss: {total_loss/(i+1):.4f} | {acc_str}", flush=True)

    return total_loss / len(loader), {k: correct[k] / total for k in model.heads.keys()}


# 학습 실행
num_epochs = 5
best_val_loss = float('inf')
save_dir = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/jiwoo/'

for epoch in range(num_epochs):
    print(f"\n[Epoch {epoch+1:02d}/{num_epochs}]")
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    train_acc_str = ' | '.join([f"{k}: {v:.4f}" for k, v in train_acc.items()])
    val_acc_str = ' | '.join([f"{k}: {v:.4f}" for k, v in val_acc.items()])
    print(f"  train loss: {train_loss:.4f} | {train_acc_str}")
    print(f"  val   loss: {val_loss:.4f} | {val_acc_str}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_dir + 'best_model.pth')
        print(f"  best model saved")


[Epoch 01/5]
  [train] batch 0/301 | loss: 4.2951 | drug_shape: 0.2500 | color_class1: 0.0312
  [train] batch 10/301 | loss: 3.7723 | drug_shape: 0.5085 | color_class1: 0.2756
  [train] batch 20/301 | loss: 3.2227 | drug_shape: 0.6280 | color_class1: 0.3586
  [train] batch 30/301 | loss: 2.8536 | drug_shape: 0.6946 | color_class1: 0.4194
  [train] batch 40/301 | loss: 2.5573 | drug_shape: 0.7386 | color_class1: 0.4703
  [train] batch 50/301 | loss: 2.3305 | drug_shape: 0.7714 | color_class1: 0.5159
  [train] batch 60/301 | loss: 2.1432 | drug_shape: 0.7951 | color_class1: 0.5471
  [train] batch 70/301 | loss: 1.9998 | drug_shape: 0.8116 | color_class1: 0.5687
  [train] batch 80/301 | loss: 1.8754 | drug_shape: 0.8272 | color_class1: 0.5926
  [train] batch 90/301 | loss: 1.7838 | drug_shape: 0.8379 | color_class1: 0.6071
  [train] batch 100/301 | loss: 1.6887 | drug_shape: 0.8478 | color_class1: 0.6247
  [train] batch 110/301 | loss: 1.5955 | drug_shape: 0.8575 | color_class1: 0.6447
 

In [19]:
# 현재 학습 중단 후 실행
# epoch 1 best model 로드 후 backbone frozen, head만 추가 학습 시도.
# 과적합 원인이 backbone에 있는지 검증 목적.

# 1. best model 로드
model.load_state_dict(torch.load(save_dir + 'best_model.pth'))
model = model.to(device)

# 2. backbone frozen
for param in model.backbone.parameters():
    param.requires_grad = False

# head만 학습되는지 확인
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터: {trainable:,} / 전체: {total:,}")

# 3. optimizer, scheduler, criterion 재정의
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=0.05
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss()

# 4. 학습 실행
num_epochs = 10
best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\n[Epoch {epoch+1:02d}/{num_epochs}]")
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    train_acc_str = ' | '.join([f"{k}: {v:.4f}" for k, v in train_acc.items()])
    val_acc_str = ' | '.join([f"{k}: {v:.4f}" for k, v in val_acc.items()])
    print(f"  train loss: {train_loss:.4f} | {train_acc_str}")
    print(f"  val   loss: {val_loss:.4f} | {val_acc_str}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_dir + 'best_model_frozen.pth')
        print(f"  best model saved")

학습 파라미터: 15,380 / 전체: 27,833,972

[Epoch 01/10]
  [train] batch 0/301 | loss: 0.3751 | drug_shape: 0.9375 | color_class1: 0.9375
  [train] batch 10/301 | loss: 0.3842 | drug_shape: 0.9801 | color_class1: 0.8977
  [train] batch 20/301 | loss: 0.3203 | drug_shape: 0.9792 | color_class1: 0.9137
  [train] batch 30/301 | loss: 0.2927 | drug_shape: 0.9758 | color_class1: 0.9284
  [train] batch 40/301 | loss: 0.2689 | drug_shape: 0.9771 | color_class1: 0.9345
  [train] batch 50/301 | loss: 0.2574 | drug_shape: 0.9792 | color_class1: 0.9387
  [train] batch 60/301 | loss: 0.2468 | drug_shape: 0.9816 | color_class1: 0.9411
  [train] batch 70/301 | loss: 0.2399 | drug_shape: 0.9824 | color_class1: 0.9437
  [train] batch 80/301 | loss: 0.2339 | drug_shape: 0.9830 | color_class1: 0.9444
  [train] batch 90/301 | loss: 0.2291 | drug_shape: 0.9832 | color_class1: 0.9440
  [train] batch 100/301 | loss: 0.2207 | drug_shape: 0.9839 | color_class1: 0.9465
  [train] batch 110/301 | loss: 0.2206 | drug_shap

KeyboardInterrupt: 

In [20]:
df_train = pd.read_csv(save_dir + 'pilliot_split_train.csv')
df_val = pd.read_csv(save_dir + 'pilliot_split_val.csv')

print("train color 분포:")
print(df_train['color_class1'].value_counts(normalize=True).round(3))
print("\nval color 분포:")
print(df_val['color_class1'].value_counts(normalize=True).round(3))

train color 분포:
color_class1
하양    0.276
노랑    0.151
분홍    0.108
갈색    0.084
주황    0.083
초록    0.081
연두    0.067
파랑    0.060
보라    0.020
청록    0.018
빨강    0.017
회색    0.012
자주    0.010
검정    0.008
투명    0.004
Name: proportion, dtype: float64

val color 분포:
color_class1
하양    0.256
분홍    0.130
노랑    0.118
연두    0.111
주황    0.101
갈색    0.094
초록    0.068
파랑    0.054
보라    0.028
청록    0.021
검정    0.021
Name: proportion, dtype: float64


#### **과적합 이슈**

- color val 성능이 0.74-0.77 수준에서 수렴 (train은 0.93-0.97까지 상승) → 과적합
- ColorJitter 제거, weight_decay 강화(0.05), backbone frozen 등 시도하였으나 개선 미미
- 과적합 원인: val에 파랑·기타 클래스 누락 및 train/val 간 분포 불균형으로 분석됨
- val에 누락된 color 클래스(빨강/회색/자주/투명)를 기타로 통합하거나, color/shape 동시 stratify split을 적용하여 train/val 분포 불균형 해소 필요